# How programs run

The loop from notebook 01, which Python ran tens of times slower than C:

In [ ]:
def sum_of_remainders(count: int) -> int:
    total = 0
    for i in range(count):
        total += i % 7
    return total

## What the Python interpreter really does

The standard Python, called **CPython** because it is itself written in C, works in two stages:

1. It first compiles your source code into **bytecode**: a list of simple instructions for an imaginary machine, not for the real CPU. This happens automatically and very quickly (the `.pyc` files in `__pycache__` folders are cached bytecode).
2. Then the **interpreter loop**, a big program written in C, reads the bytecode instructions **one at a time** and carries each of them out.

So Python is still a "live interpreter": the translation from bytecode to real work happens again every time an instruction runs. The `dis` module ("disassembler") shows the bytecode of our function:

In [ ]:
import dis

dis.dis(sum_of_remainders)

  1           RESUME                   0

  2           LOAD_SMALL_INT           0
              STORE_FAST               1 (total)

  3           LOAD_GLOBAL              1 (range + NULL)
              LOAD_FAST_BORROW         0 (count)
              CALL                     1
              GET_ITER
      L1:     FOR_ITER                18 (to L2)
              STORE_FAST               2 (i)

  4           LOAD_FAST_BORROW_LOAD_FAST_BORROW 18 (total, i)
              LOAD_SMALL_INT           7
              BINARY_OP                6 (%)
              BINARY_OP               13 (+=)
              STORE_FAST               1 (total)
              JUMP_BACKWARD           20 (to L1)

  3   L2:     END_FOR
              POP_ITER

  5           LOAD_FAST_BORROW         1 (total)
              RETURN_VALUE


Each line is one bytecode instruction. The lines between `FOR_ITER` and `JUMP_BACKWARD` form the body of the loop, so they run ten million times. For **each** of them, the interpreter must:

- fetch the instruction and work out which one it is,
- look up the values it works on,
- check what **type** they are (integer? decimal? text?), because the same `BINARY_OP` means different things for numbers and for strings,
- only then do the actual arithmetic.

The compiled C loop skips all that bookkeeping: its body is a handful of machine instructions that the CPU executes directly.

In [ ]:
import subprocess

from compare_efficiency.project_paths import BUILD_DIRECTORY, C_DIRECTORY

BUILD_DIRECTORY.mkdir(exist_ok=True)

## Types: checked before running, or while running

A **type** says what kind of value something is: an integer, a decimal number, a piece of text… Languages differ in *when* they check types:

- C is **statically typed**: the type of every variable and function argument is written in the code, and the compiler checks them all *before* the program runs.
- Python is **dynamically typed**: a variable is just a name that can refer to any value, and the types are checked *while* the program runs, at the moment each operation happens.

This small C program calls a function that expects a decimal number (`double`) with a piece of text instead:

In [ ]:
print((C_DIRECTORY / "type_error.c").read_text())

#include <stdio.h>

double half(double number) {
    return number / 2;
}

int main(void) {
    printf("%f\n", half("ten"));
    return 0;
}



In [ ]:
failed_compilation = subprocess.run(["gcc", str(C_DIRECTORY / "type_error.c"), "-o", str(BUILD_DIRECTORY / "type_error")], capture_output=True, text=True)
print(failed_compilation.stderr)

/home/avidaldo/TRABAJO/wip-clase/PIA-SAA/example_repos/compare-python-c-efficiency/c/type_error.c: In function ‘main’:
/home/avidaldo/TRABAJO/wip-clase/PIA-SAA/example_repos/compare-python-c-efficiency/c/type_error.c:8:25: error: incompatible type for argument 1 of ‘half’
    8 |     printf("%f\n", half("ten"));
      |                         ^~~~~
      |                         |
      |                         char *
/home/avidaldo/TRABAJO/wip-clase/PIA-SAA/example_repos/compare-python-c-efficiency/c/type_error.c:3:20: note: expected ‘double’ but argument is of type ‘char *’
    3 | double half(double number) {
      |             ~~~~~~~^~~~~~



The compiler refuses to produce an executable. In C, text is represented as `char *` (the memory address of a sequence of characters), and the compiler reports that `half` expected a `double`. The mistake is caught **before the program ever runs**.

The same mistake in Python goes unnoticed until the faulty line actually executes. Here it hides in a branch that normal runs never reach:

In [ ]:
def half(number):
    return number / 2

def halve_all(values, include_label=False):
    halves = [half(value) for value in values]
    if include_label:
        halves.append(half("ten"))
    return halves

halve_all([1, 2, 3])

[0.5, 1.0, 1.5]

The program works, until someone uses the rarely used option:

In [ ]:
try:
    halve_all([1, 2, 3], include_label=True)
except TypeError as error:
    caught_error = error
caught_error

TypeError("unsupported operand type(s) for /: 'str' and 'int'")

Dynamic typing has real advantages: code is shorter, more flexible, and you can try things interactively without declaring anything. But it has two costs:

- some errors are only found when the faulty line runs (which is one reason why testing matters so much in Python),
- every operation must check types while the program runs, which is part of why Python is slow.

## Hybrids: bytecode and JIT compilation

In practice the boundary is blurry:

- **Java** and **C#** are compiled to bytecode, which runs on a virtual machine.
- That virtual machine includes a **JIT compiler** (*just-in-time*): while the program runs, it detects the parts that run most often and compiles them into machine code on the fly. JavaScript engines in web browsers do the same.
- Classic compilation, done completely before the program runs, is called **AOT** (*ahead-of-time*) compilation.

"Compiled" and "interpreted" really describe *implementations*, not languages. CPython interprets Python, but other implementations of Python use a JIT, and some tools compile parts of a Python program ahead of time. The deep dive [Compiling Python](compiling_python.ipynb) uses several of them.

## Looking at machine code

Machine code is just numbers, but it has a human-readable form called **assembly**: one line per instruction. `gcc -S` stops after translating to assembly and shows it. Here is a one-line C function that adds two integers:

In [ ]:
add_source = "long long add(long long a, long long b) { return a + b; }"
assembly = subprocess.run(["gcc", "-O2", "-S", "-x", "c", "-", "-o", "-"], input=add_source,
                          capture_output=True, text=True, check=True).stdout
assembly_lines = assembly.splitlines()
# macOS prefixes function names with an underscore: "_add:"
function_start = next(index for index, line in enumerate(assembly_lines) if line.endswith("add:"))
function_end = next(index for index, line in enumerate(assembly_lines) if line.strip() == "ret")
print("\n".join(line for line in assembly_lines[function_start:function_end + 1] if not line.lstrip().startswith(".")))

add:
	endbr64
	leaq	(%rdi,%rsi), %rax
	ret


Only the function itself is shown, without the directives for the assembler (lines starting with a dot). On an x86-64 processor it contains:

- `endbr64`: a security marker that modern compilers add at the start of functions. It does no arithmetic.
- `leaq (%rdi,%rsi), %rax`: adds the two arguments and stores the result. The arguments arrive in CPU **registers** (`%rdi`, `%rsi`), tiny storage slots inside the processor itself, and the result goes in `%rax`.
- `ret`: returns to the caller.

On an ARM processor the instruction names differ, but the function is just as short.

Compare it with the bytecode of the same function in Python. It has a similar number of instructions, but each bytecode instruction triggers dozens or hundreds of machine instructions inside the interpreter loop.

In [ ]:
def add(a, b):
    return a + b

dis.dis(add)

  1           RESUME                   0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE


## Key ideas

- CPython first compiles source code to **bytecode**, then its interpreter loop executes the bytecode one instruction at a time, checking the types of the values at every operation.
- **Static typing** (C) catches type errors before the program runs; **dynamic typing** (Python) finds them only when the faulty line executes, but makes code shorter and more flexible.
- **JIT** compilers compile the most-used parts of a program while it runs; **AOT** compilation translates everything in advance. These describe implementations, not languages.
- Machine code has a readable form, **assembly**: a simple C function becomes a handful of instructions.